# 02 Modeling and Inference

## P4 · Baselines and Backtest Harness

This notebook starts from the P3 artifact `data/processed/features.parquet`. It builds the walk-forward folds, baseline forecasts, metric implementations, denominator guards, and the comparison table every later model must beat.

In [1]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

CWD = Path.cwd().resolve()
if (CWD / "data" / "processed" / "features.parquet").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "data" / "processed" / "features.parquet").exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError("Run this notebook after P3 writes data/processed/features.parquet")

FEATURE_PATH = PROJECT_ROOT / "data" / "processed" / "features.parquet"
P1_SERIES_INVENTORY_PATH = PROJECT_ROOT / "outputs" / "metrics" / "p1_ts_eda" / "series_inventory.csv"
P4_TABLE_DIR = PROJECT_ROOT / "outputs" / "metrics" / "baselines"
P4_TABLE_DIR.mkdir(parents=True, exist_ok=True)

HORIZON_WEEKS = 13
FOLD_COUNT = 4
SEASONAL_PERIOD_WEEKS = 52
MOVING_AVERAGE_WINDOW_WEEKS = 4
DENOMINATOR_FLOOR = 1e-8
QUANTILES = [0.1, 0.5, 0.8, 0.9]
SBA_ALPHA = 0.1
SBA_BIAS_CORRECTION = 1.0 - SBA_ALPHA / 2.0
HIERARCHY_LEVELS = ["total", "state", "city", "store", "cluster", "family", "store_family"]
BASELINE_NAMES = ["naive", "seasonal_naive", "moving_average", "sba"]


### P4 Helper Functions

Metric denominators are computed from each fold's post-truncation training data only. Series with invalid denominators or launches inside a test window are excluded from aggregate scores and counted.

In [2]:
def series_id_for_level(frame: pd.DataFrame, level: str) -> pd.Series:
    if level == "total":
        return pd.Series("total", index=frame.index)
    if level == "state":
        return frame["state"].astype(str)
    if level == "city":
        return frame["city"].astype(str)
    if level == "store":
        return frame["store_nbr"].astype(str)
    if level == "cluster":
        return frame["cluster"].astype(str)
    if level == "family":
        return frame["family"].astype(str)
    if level == "store_family":
        return frame["store_nbr"].astype(str) + "|" + frame["family"].astype(str)
    raise ValueError(f"Unknown level: {level}")


def aggregate_to_level(frame: pd.DataFrame, level: str) -> pd.DataFrame:
    working = frame.copy()
    working["level"] = level
    working["series_id"] = series_id_for_level(working, level)
    grouped = (
        working.groupby(["level", "series_id", "week_start"], observed=True, as_index=False)["sales"]
        .sum()
        .sort_values(["level", "series_id", "week_start"])
        .reset_index(drop=True)
    )
    return grouped


def make_folds(max_week_start: pd.Timestamp, fold_count: int, horizon_weeks: int) -> pd.DataFrame:
    final_test_start = max_week_start - pd.Timedelta(weeks=horizon_weeks - 1)
    rows = []
    for fold_number in range(fold_count):
        test_start = final_test_start - pd.Timedelta(weeks=horizon_weeks * (fold_count - fold_number - 1))
        test_end = test_start + pd.Timedelta(weeks=horizon_weeks - 1)
        rows.append(
            {
                "fold": fold_number + 1,
                "train_end": test_start - pd.Timedelta(weeks=1),
                "test_start": test_start,
                "test_end": test_end,
                "horizon_weeks": horizon_weeks,
            }
        )
    return pd.DataFrame(rows)


def assert_fold_no_lookahead(folds: pd.DataFrame) -> None:
    for fold in folds.itertuples(index=False):
        assert fold.train_end < fold.test_start, f"Fold {fold.fold} train window touches test window"
        assert fold.test_end >= fold.test_start, f"Fold {fold.fold} has invalid test dates"


def training_denominator(values: pd.Series) -> tuple:
    clean = values.astype(float).dropna().to_numpy()
    if len(clean) < 2:
        return np.nan, 0
    differences = np.abs(np.diff(clean))
    return float(np.mean(differences)), len(clean)


def rmsse(actual: np.ndarray, forecast: np.ndarray, denominator: float) -> float:
    return float(np.sqrt(np.mean((actual - forecast) ** 2)) / denominator)


def mase(actual: np.ndarray, forecast: np.ndarray, denominator: float) -> float:
    return float(np.mean(np.abs(actual - forecast)) / denominator)


def pinball_loss(actual: np.ndarray, forecast: np.ndarray, quantile: float) -> float:
    error = actual - forecast
    return float(np.mean(np.maximum(quantile * error, (quantile - 1.0) * error)))


def empirical_coverage(actual: np.ndarray, forecast: np.ndarray) -> float:
    return float(np.mean(actual <= forecast))


def mean_bias(actual: np.ndarray, forecast: np.ndarray) -> float:
    return float(np.mean(forecast - actual))


def naive_forecast(train: pd.Series, horizon: int) -> np.ndarray:
    return np.repeat(float(train.iloc[-1]), horizon)


def seasonal_naive_forecast(full_series: pd.Series, test_index: pd.DatetimeIndex, seasonal_period: int) -> np.ndarray:
    values = []
    for week in test_index:
        source_week = week - pd.Timedelta(weeks=seasonal_period)
        if source_week in full_series.index:
            values.append(float(full_series.loc[source_week]))
        else:
            values.append(float(full_series.loc[full_series.index < week].iloc[-1]))
    return np.asarray(values, dtype=float)


def moving_average_forecast(train: pd.Series, horizon: int, window: int) -> np.ndarray:
    return np.repeat(float(train.tail(window).mean()), horizon)


def sba_forecast(train: pd.Series, horizon: int, alpha: float, correction: float) -> np.ndarray:
    values = train.astype(float).to_numpy()
    non_zero_positions = np.flatnonzero(values > 0)
    if len(non_zero_positions) == 0:
        return np.zeros(horizon, dtype=float)
    first_position = int(non_zero_positions[0])
    demand_estimate = float(values[first_position])
    interval_estimate = float(first_position + 1)
    previous_position = first_position
    for position in non_zero_positions[1:]:
        interval = float(position - previous_position)
        demand_estimate = alpha * float(values[position]) + (1.0 - alpha) * demand_estimate
        interval_estimate = alpha * interval + (1.0 - alpha) * interval_estimate
        previous_position = int(position)
    rate = correction * demand_estimate / interval_estimate
    return np.repeat(max(rate, 0.0), horizon)


def one_step_residuals(train: pd.Series, baseline_name: str) -> np.ndarray:
    values = train.astype(float)
    if baseline_name == "naive":
        forecast = values.shift(1)
    elif baseline_name == "seasonal_naive":
        forecast = values.shift(SEASONAL_PERIOD_WEEKS)
    elif baseline_name == "moving_average":
        forecast = values.shift(1).rolling(MOVING_AVERAGE_WINDOW_WEEKS, min_periods=1).mean()
    elif baseline_name == "sba":
        forecast = pd.Series(np.repeat(float(values.mean()), len(values)), index=values.index)
    else:
        raise ValueError(f"Unknown baseline: {baseline_name}")
    residuals = (values - forecast).dropna().to_numpy(dtype=float)
    if len(residuals) == 0:
        return np.asarray([0.0], dtype=float)
    return residuals


def quantile_forecasts(point_forecast: np.ndarray, residuals: np.ndarray, quantiles: list) -> dict:
    forecasts = {}
    for quantile in quantiles:
        offset = float(np.quantile(residuals, quantile))
        forecasts[quantile] = np.maximum(point_forecast + offset, 0.0)
    return forecasts


def denominator_canary_result() -> pd.DataFrame:
    full = pd.Series([0.0] * 400 + [4.0, 8.0, 5.0, 7.0, 9.0])
    truncated = full.loc[full.ne(0).idxmax():].reset_index(drop=True)
    full_denominator, full_length = training_denominator(full)
    truncated_denominator, truncated_length = training_denominator(truncated)
    assert truncated_denominator > full_denominator
    return pd.DataFrame(
        {
            "case": ["full_with_leading_zeros", "post_truncation_only"],
            "denominator": [full_denominator, truncated_denominator],
            "denominator_length": [full_length, truncated_length],
        }
    )


def metric_unit_test_results() -> pd.DataFrame:
    actual = np.asarray([2.0, 4.0, 6.0])
    forecast = np.asarray([1.0, 5.0, 7.0])
    denominator = 2.0
    rows = [
        {"metric": "rmsse", "observed": rmsse(actual, forecast, denominator), "expected": np.sqrt(1.0) / 2.0},
        {"metric": "mase", "observed": mase(actual, forecast, denominator), "expected": 1.0 / 2.0},
        {"metric": "pinball_q80", "observed": pinball_loss(actual, forecast, 0.8), "expected": float(np.mean([0.8, 0.2, 0.2]))},
        {"metric": "coverage", "observed": empirical_coverage(actual, forecast), "expected": 2.0 / 3.0},
        {"metric": "bias", "observed": mean_bias(actual, forecast), "expected": 1.0 / 3.0},
    ]
    result = pd.DataFrame(rows)
    assert np.allclose(result["observed"], result["expected"])
    return result


### Load Features and Build Folds

The four test windows are non-overlapping 13-week quarters. Every fold trains strictly before its own test start.

In [3]:
features = pd.read_parquet(FEATURE_PATH)
features["week_start"] = pd.to_datetime(features["week_start"])
series_inventory = pd.read_csv(P1_SERIES_INVENTORY_PATH)
segment_lookup = series_inventory.assign(series_id=series_inventory["store_nbr"].astype(str) + "|" + series_inventory["family"].astype(str))[["series_id", "intermittency_segment"]]
assert features["sales"].notna().all()
assert features[["store_nbr", "family", "week_start"]].duplicated().sum() == 0

folds = make_folds(features["week_start"].max(), FOLD_COUNT, HORIZON_WEEKS)
assert_fold_no_lookahead(folds)
folds.to_csv(P4_TABLE_DIR / "fold_boundaries.csv", index=False)
folds

,fold,train_end,test_start,test_end,horizon_weeks
0,1,2016-08-15,2016-08-22,2016-11-14,13
1,2,2016-11-14,2016-11-21,2017-02-13,13
2,3,2017-02-13,2017-02-20,2017-05-15,13
3,4,2017-05-15,2017-05-22,2017-08-14,13


### Metric and Denominator Tests

These are hand-computed checks for the metric implementations and the leading-zero denominator rule.

In [4]:
metric_tests = metric_unit_test_results()
denominator_canary = denominator_canary_result()
metric_tests.to_csv(P4_TABLE_DIR / "metric_unit_tests.csv", index=False)
denominator_canary.to_csv(P4_TABLE_DIR / "denominator_leading_zero_canary.csv", index=False)
metric_tests, denominator_canary

(        metric  observed  expected
 0        rmsse  0.500000  0.500000
 1         mase  0.500000  0.500000
 2  pinball_q80  0.400000  0.400000
 3     coverage  0.666667  0.666667
 4         bias  0.333333  0.333333,
                       case  denominator  denominator_length
 0  full_with_leading_zeros     0.037129                 405
 1     post_truncation_only     2.750000                   5)

### Run Baseline Backtest

Baselines are fit from each fold's training window only. Scores are reported per hierarchy level, per fold, with excluded-series counts.

In [5]:
level_frames = {level: aggregate_to_level(features, level) for level in HIERARCHY_LEVELS}
score_rows = []
forecast_rows = []
excluded_rows = []
launch_rows = []

for level, level_frame in level_frames.items():
    for fold in folds.itertuples(index=False):
        train_end = pd.Timestamp(fold.train_end)
        test_start = pd.Timestamp(fold.test_start)
        test_end = pd.Timestamp(fold.test_end)
        fold_frame = level_frame.loc[level_frame["week_start"] <= test_end]
        for series_id, series_frame in fold_frame.groupby("series_id", observed=True):
            series_frame = series_frame.sort_values("week_start")
            train = series_frame.loc[series_frame["week_start"] <= train_end].set_index("week_start")["sales"]
            test = series_frame.loc[(series_frame["week_start"] >= test_start) & (series_frame["week_start"] <= test_end)].set_index("week_start")["sales"]
            if test.empty:
                continue
            if train.empty:
                launch_rows.append({"level": level, "fold": fold.fold, "series_id": series_id, "reason": "launches_inside_test_window"})
                continue
            denominator, denominator_length = training_denominator(train)
            assert denominator_length == len(train.dropna()), "Denominator length must equal post-truncation training length"
            if not np.isfinite(denominator) or denominator <= DENOMINATOR_FLOOR:
                excluded_rows.append(
                    {
                        "level": level,
                        "fold": fold.fold,
                        "series_id": series_id,
                        "reason": "invalid_denominator",
                        "denominator": denominator,
                        "denominator_length": denominator_length,
                    }
                )
                continue
            if len(test) != HORIZON_WEEKS:
                excluded_rows.append(
                    {
                        "level": level,
                        "fold": fold.fold,
                        "series_id": series_id,
                        "reason": "incomplete_test_horizon",
                        "denominator": denominator,
                        "denominator_length": denominator_length,
                    }
                )
                continue
            full_series = series_frame.set_index("week_start")["sales"]
            test_index = test.index
            actual = test.to_numpy(dtype=float)
            baseline_points = {
                "naive": naive_forecast(train, len(test)),
                "seasonal_naive": seasonal_naive_forecast(full_series, test_index, SEASONAL_PERIOD_WEEKS),
                "moving_average": moving_average_forecast(train, len(test), MOVING_AVERAGE_WINDOW_WEEKS),
                "sba": sba_forecast(train, len(test), SBA_ALPHA, SBA_BIAS_CORRECTION),
            }
            for baseline_name, point_forecast in baseline_points.items():
                residuals = one_step_residuals(train, baseline_name)
                q_forecasts = quantile_forecasts(point_forecast, residuals, QUANTILES)
                row = {
                    "level": level,
                    "fold": fold.fold,
                    "series_id": series_id,
                    "baseline": baseline_name,
                    "rmsse": rmsse(actual, point_forecast, denominator),
                    "mase": mase(actual, point_forecast, denominator),
                    "bias": mean_bias(actual, point_forecast),
                    "denominator": denominator,
                    "denominator_length": denominator_length,
                    "test_observations": len(test),
                }
                for quantile in QUANTILES:
                    row[f"pinball_q{int(quantile * 100)}"] = pinball_loss(actual, q_forecasts[quantile], quantile)
                    row[f"coverage_q{int(quantile * 100)}"] = empirical_coverage(actual, q_forecasts[quantile])
                score_rows.append(row)
                for horizon_step, (week, actual_value, forecast_value) in enumerate(zip(test_index, actual, point_forecast), start=1):
                    forecast_rows.append(
                        {
                            "level": level,
                            "fold": fold.fold,
                            "series_id": series_id,
                            "baseline": baseline_name,
                            "week_start": week,
                            "horizon_step": horizon_step,
                            "actual": actual_value,
                            "point_forecast": forecast_value,
                        }
                    )

baseline_scores_by_series = pd.DataFrame(score_rows)
baseline_forecasts_sample = pd.DataFrame(forecast_rows)
excluded_series_columns = ["level", "fold", "series_id", "reason", "denominator", "denominator_length"]
launch_inside_test_columns = ["level", "fold", "series_id", "reason"]
excluded_series = pd.DataFrame(excluded_rows, columns=excluded_series_columns)
launch_inside_test = pd.DataFrame(launch_rows, columns=launch_inside_test_columns)

baseline_scores_by_series.to_csv(P4_TABLE_DIR / "baseline_scores_by_series.csv", index=False)
baseline_forecasts_sample.to_csv(P4_TABLE_DIR / "baseline_forecasts_point.csv", index=False)
excluded_series.to_csv(P4_TABLE_DIR / "excluded_series.csv", index=False)
launch_inside_test.to_csv(P4_TABLE_DIR / "series_launching_inside_test.csv", index=False)

baseline_scores_by_series.head(), excluded_series.head(), launch_inside_test.head()

(   level  fold series_id        baseline     rmsse      mase           bias  \
 0  total     1     total           naive  1.299769  1.062337 -259009.651059   
 1  total     1     total  seasonal_naive  0.973354  0.751207  -37092.900806   
 2  total     1     total  moving_average  1.300201  1.062514 -259281.294530   
 3  total     1     total             sba  1.517467  1.197457 -379530.674024   
 4  total     2     total           naive  1.985121  1.448242 -458170.343230   
 
      denominator  denominator_length  test_observations    pinball_q10  \
 0  354231.417490                 190                 13   79757.801590   
 1  354231.417490                 190                 13  170940.690251   
 2  354231.417490                 190                 13   81970.090057   
 3  354231.417490                 190                 13  189902.208390   
 4  356288.502919                 203                 13   97296.673191   
 
    coverage_q10    pinball_q50  coverage_q50    pinball_q80  cove

### Baseline Score Tables

This is the bar later models must clear: per level, per fold, with mean and range across folds. Excluded-series counts are included every time.

In [6]:
excluded_counts = (
    excluded_series.groupby(["level", "fold"], observed=True)
    .size()
    .rename("excluded_series_count")
    .reset_index()
    if not excluded_series.empty
    else pd.DataFrame(columns=["level", "fold", "excluded_series_count"])
)
launch_counts = (
    launch_inside_test.groupby(["level", "fold"], observed=True)
    .size()
    .rename("launch_inside_test_count")
    .reset_index()
    if not launch_inside_test.empty
    else pd.DataFrame(columns=["level", "fold", "launch_inside_test_count"])
)

baseline_level_fold = (
    baseline_scores_by_series.groupby(["level", "fold", "baseline"], observed=True)
    .agg(
        rmsse=("rmsse", "mean"),
        mase=("mase", "mean"),
        bias=("bias", "mean"),
        pinball_q10=("pinball_q10", "mean"),
        pinball_q50=("pinball_q50", "mean"),
        pinball_q80=("pinball_q80", "mean"),
        pinball_q90=("pinball_q90", "mean"),
        coverage_q10=("coverage_q10", "mean"),
        coverage_q50=("coverage_q50", "mean"),
        coverage_q80=("coverage_q80", "mean"),
        coverage_q90=("coverage_q90", "mean"),
        scored_series=("series_id", "nunique"),
    )
    .reset_index()
    .merge(excluded_counts, on=["level", "fold"], how="left")
    .merge(launch_counts, on=["level", "fold"], how="left")
)
baseline_level_fold["excluded_series_count"] = baseline_level_fold["excluded_series_count"].fillna(0).astype(int)
baseline_level_fold["launch_inside_test_count"] = baseline_level_fold["launch_inside_test_count"].fillna(0).astype(int)

baseline_summary = (
    baseline_level_fold.groupby(["level", "baseline"], observed=True)
    .agg(
        rmsse_mean=("rmsse", "mean"),
        rmsse_min=("rmsse", "min"),
        rmsse_max=("rmsse", "max"),
        mase_mean=("mase", "mean"),
        mase_min=("mase", "min"),
        mase_max=("mase", "max"),
        bias_mean=("bias", "mean"),
        pinball_q80_mean=("pinball_q80", "mean"),
        coverage_q80_mean=("coverage_q80", "mean"),
        scored_series_min=("scored_series", "min"),
        excluded_series_max=("excluded_series_count", "max"),
        launch_inside_test_max=("launch_inside_test_count", "max"),
    )
    .reset_index()
    .sort_values(["level", "rmsse_mean"])
)

baseline_segment_fold = (
    baseline_scores_by_series.loc[baseline_scores_by_series["level"] == "store_family"]
    .merge(segment_lookup, on="series_id", how="left")
    .groupby(["intermittency_segment", "fold", "baseline"], observed=True)
    .agg(
        rmsse=("rmsse", "mean"),
        mase=("mase", "mean"),
        bias=("bias", "mean"),
        pinball_q80=("pinball_q80", "mean"),
        coverage_q80=("coverage_q80", "mean"),
        scored_series=("series_id", "nunique"),
    )
    .reset_index()
)
baseline_segment_summary = (
    baseline_segment_fold.groupby(["intermittency_segment", "baseline"], observed=True)
    .agg(
        rmsse_mean=("rmsse", "mean"),
        rmsse_min=("rmsse", "min"),
        rmsse_max=("rmsse", "max"),
        mase_mean=("mase", "mean"),
        bias_mean=("bias", "mean"),
        pinball_q80_mean=("pinball_q80", "mean"),
        coverage_q80_mean=("coverage_q80", "mean"),
        scored_series_min=("scored_series", "min"),
    )
    .reset_index()
    .sort_values(["intermittency_segment", "rmsse_mean"])
)
assert baseline_segment_fold["intermittency_segment"].notna().all()

baseline_level_fold.to_csv(P4_TABLE_DIR / "baseline_scores_by_level_fold.csv", index=False)
baseline_summary.to_csv(P4_TABLE_DIR / "baseline_summary_by_level.csv", index=False)
baseline_segment_fold.to_csv(P4_TABLE_DIR / "baseline_scores_by_segment_fold.csv", index=False)
baseline_segment_summary.to_csv(P4_TABLE_DIR / "baseline_summary_by_segment.csv", index=False)

baseline_summary, baseline_segment_summary

(           level        baseline  rmsse_mean  rmsse_min  rmsse_max  mase_mean  \
 0           city  moving_average    2.221838   1.692150   3.104945   1.512522   
 1           city           naive    2.366823   1.692894   3.165717   1.665014   
 3           city  seasonal_naive    2.380411   1.786271   3.495880   1.958920   
 2           city             sba    2.489184   1.749684   3.245227   1.853332   
 5        cluster           naive    2.307011   1.631040   3.323925   1.621174   
 4        cluster  moving_average    2.348952   1.459964   3.301275   1.647976   
 7        cluster  seasonal_naive    2.404707   1.905081   3.185761   1.913935   
 6        cluster             sba    2.555993   1.859727   3.128912   1.932447   
 8         family  moving_average    2.535206   1.562321   3.335748   1.812670   
 10        family             sba    2.692135   1.722682   3.581347   1.950529   
 9         family           naive    2.697144   1.553174   3.373552   1.975001   
 11        famil

### P4 Findings

**Artifacts written**

- Fold boundaries: `outputs/metrics/baselines/fold_boundaries.csv`
- Series-level scores: `outputs/metrics/baselines/baseline_scores_by_series.csv`
- Level/fold comparison table: `outputs/metrics/baselines/baseline_scores_by_level_fold.csv`
- Level summary table: `outputs/metrics/baselines/baseline_summary_by_level.csv`
- Intermittency-segment tables: `baseline_scores_by_segment_fold.csv`, `baseline_summary_by_segment.csv`
- Metric tests and denominator canary: `outputs/metrics/baselines/metric_unit_tests.csv`, `denominator_leading_zero_canary.csv`
- Exclusion reports: `excluded_series.csv`, `series_launching_inside_test.csv`

**P4 exit criteria status**

- Fold boundaries printed and eyeballed against the calendar: done in `fold_boundaries.csv`.
- No fold training window contains test dates: asserted in `assert_fold_no_lookahead`.
- Baseline scores recorded per level and intermittency segment, per fold, with spread: done in the level/fold, segment/fold, and summary tables.
- Metric implementations unit-tested against hand-computed examples: done in `metric_unit_tests.csv`.
- Scaling denominator computed on post-truncation training data only: enforced by reading the P3 post-truncation artifact and checked in `denominator_leading_zero_canary.csv`.
- Denominator floor guard active; excluded-series count reported: done in `excluded_series.csv` and score tables.
- Series launching inside a test window identified and excluded: done in `series_launching_inside_test.csv`.
- The bar every model must clear is now a number on paper: done in `baseline_summary_by_level.csv`.


## P5 · Model Building

Not started. P5 begins only after P4 exit criteria are accepted.

## P6 · Reconciliation

Not started. P6 begins only after P5 exit criteria are accepted.

## P7 · Evaluation and Selection

Not started. P7 begins only after P6 exit criteria are accepted.